# Example: MMB-by-MMB on Simulated Data

## 1. Imports and random seed

In [35]:
import random
import numpy as np
import pandas as pd

from generator.random_graph import simulate_dag, set_latent_nodes
from generator.Continuous_model import linear_sem
from generator.Discrete_model import discrete_model
from mmb_by_mmb import data_MMB_by_MMB, oracle_MMB_by_MMB
from dagtopag.dag2 import dag2pag
from Utils.PartMixGraph import Mark
from Utils.util_tools import local_mark_evaluation

np.random.seed(42)
random.seed(42)


def build_dag_with_latent(num_nodes=12, expected_degree=3, graph_type="ER", num_latent=1, max_tries=100):
    for _ in range(max_tries):
        dag = simulate_dag(
            num_nodes=num_nodes,
            expected_degree=expected_degree,
            graph_type=graph_type,
        )
        dag_with_latent, latent_nodes = set_latent_nodes(dag.copy(), num_latent=num_latent)
        if len(latent_nodes) == num_latent:
            return dag_with_latent, latent_nodes
    raise RuntimeError("Failed to generate a DAG with the specified number of latent nodes.")

## 2. Generate a DAG and set latent nodes

In [36]:
dag, latent_nodes = build_dag_with_latent(
    num_nodes=20,
    expected_degree=3,
    graph_type="ER",
    num_latent=2,
)
observed_vars = [col for col in dag.columns if col not in latent_nodes]

print("Latent nodes:", latent_nodes)
print("Number of observed variables:", len(observed_vars))
dag

Latent nodes: ['L4', 'L3']
Number of observed variables: 18


,V1,V2,L3,L4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
V1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
V2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
L3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
L4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
V5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
V6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
V7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
V8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
V9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
V10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Build the true PAG

In [37]:
true_pag = dag2pag(dag, latent_nodes=latent_nodes)["PAG.DataFrame"]
true_pag

,V1,V2,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
V1,0,0,0,1,0,0,0,0,0,0,0,0,2,2,2,0,2,2
V2,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0
V5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
V6,1,0,0,0,0,2,0,2,0,0,0,0,2,0,0,0,0,0
V7,0,0,0,0,0,0,1,2,2,2,0,0,2,0,0,0,0,0
V8,0,0,0,1,0,0,0,0,0,2,0,0,0,0,0,0,0,0
V9,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0
V10,0,0,0,1,1,0,0,0,0,2,0,0,0,0,0,0,0,0
V11,0,1,0,0,1,0,0,0,0,0,2,2,0,2,0,0,0,0
V12,0,0,0,0,3,2,0,3,0,0,0,0,0,0,0,2,0,0


## 4. Choose an observed target from the true PAG

In [38]:
candidate_nodes = []
for node in true_pag.columns:
    neighbors = true_pag.index[true_pag[node] != 0].tolist()
    if len(neighbors) >= 2:
        has_spouse = any(
            true_pag.loc[neighbor, node] == Mark.ARROW.value and true_pag.loc[node, neighbor] == Mark.ARROW.value
            for neighbor in neighbors
        )
        no_non_directed_edges = all(
            true_pag.loc[neighbor, node] != Mark.CIRCLE.value and true_pag.loc[node, neighbor] != Mark.CIRCLE.value
            for neighbor in neighbors
        )
        if has_spouse or no_non_directed_edges:
            candidate_nodes.append(node)

nodes_more_than_two_neighbors = []
for node in true_pag.columns:
    neighbors = true_pag.index[true_pag[node] != 0].tolist()
    if len(neighbors) > 2:
        nodes_more_than_two_neighbors.append(node)

target_pool = candidate_nodes or nodes_more_than_two_neighbors
target = random.choice(target_pool)
print("Candidate targets:", candidate_nodes)
print("Target:", target)

Candidate targets: ['V8', 'V12', 'V17', 'V18', 'V19', 'V20']
Target: V19


## 5. Generate continuous data

In [39]:
continuous_data = linear_sem(dag, sample_size=1000, noise_scale=1.0)
continuous_observed_data = continuous_data[observed_vars].copy()
continuous_observed_data.head()

,V1,V2,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
0,0.902255,0.841493,-0.259976,2.441513,-1.897673,2.970380,1.089661,1.018754,2.661324,-0.360916,2.952645,-1.418144,1.520949,-5.821537,-0.664334,-8.177744,-1.193716,-5.125824
1,-2.011918,-1.165443,-0.869305,-1.395223,0.113485,-2.714737,-0.034626,0.776559,-1.933364,0.092551,-1.525466,2.372471,-4.749819,5.476791,4.501200,6.319574,1.059123,5.382511
2,1.018887,1.051950,0.111897,0.651326,1.795277,1.708362,-3.552032,-1.706944,-0.497176,-0.403945,-0.942707,0.556818,0.850207,5.029962,0.887243,5.464320,2.029269,2.650537
3,-0.050119,-0.978955,0.103279,0.176211,1.492533,-1.427769,-3.378628,-1.728505,-2.564545,-0.690022,-1.597279,3.397243,-4.493611,10.569443,8.645180,14.482191,8.757251,8.236830
4,-2.529337,0.810704,1.518624,-1.338861,-0.308320,-1.920337,2.230822,0.090817,-0.571278,1.865206,1.643688,1.300708,-3.579483,2.413006,2.866088,0.227382,-0.811558,2.139941


## 6. Learn from the continuous observed data and evaluate

In [40]:
continuous_result = data_MMB_by_MMB(
    continuous_observed_data,
    target=target,
    ci_method="fisherz",
    mb_method="gaussian_MB",
    max_depth=3,
    alpha=0.01,
    max_K=5
)
continuous_pred_pag = continuous_result["PAG.DataFrame"]
continuous_eval = local_mark_evaluation(continuous_pred_pag, true_pag, target)
continuous_summary = pd.Series(
    {
        **continuous_eval,
        "CI_num": continuous_result["CI_num"],
        "runtime_sec": continuous_result["runtime_sec"],
    },
    name="continuous",
)
continuous_summary.to_frame()

,continuous
Local-SHD,2.000000
Mark-Precision,0.500000
Mark-Recall,0.500000
Mark-F1,0.500000
CI_num,801.000000
runtime_sec,0.044922


In [41]:
continuous_result_complete = data_MMB_by_MMB(
    continuous_observed_data,
    target=target,
    ci_method="fisherz",
    mb_method="gaussian_MB",
    max_depth=3,
    alpha=0.01,
    whether_fast=False,
    max_K=5
)
continuous_pred_pag_complete = continuous_result_complete["PAG.DataFrame"]
continuous_eval_complete = local_mark_evaluation(continuous_pred_pag_complete, true_pag, target)
continuous_summary = pd.Series(
    {
        **continuous_eval_complete,
        "CI_num": continuous_result_complete["CI_num"],
        "runtime_sec": continuous_result_complete["runtime_sec"],
    },
    name="continuous",
)
continuous_summary.to_frame()

,continuous
Local-SHD,1.000000
Mark-Precision,0.750000
Mark-Recall,0.750000
Mark-F1,0.750000
CI_num,2448.000000
runtime_sec,0.097271


## 7. Generate discrete data

In [42]:
discrete_data = discrete_model(dag, num_categories=3, sample_size=1000)
discrete_observed_data = discrete_data[observed_vars].copy()
discrete_observed_data.head()

,V1,V2,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20
0,1,2,2,0,1,0,1,1,1,1,2,0,1,1,2,1,2,2
1,1,1,0,0,0,0,2,2,1,0,1,1,0,2,0,2,1,2
2,1,1,0,0,1,0,0,0,0,1,0,1,2,2,0,1,2,2
3,0,0,0,0,1,1,0,0,2,1,0,2,1,2,1,2,2,2
4,1,1,0,0,2,0,2,1,2,2,1,1,0,0,0,1,2,2


## 8. Learn from the discrete observed data and evaluate

In [43]:
discrete_result = data_MMB_by_MMB(
    discrete_observed_data,
    target=target,
    ci_method="chisq",
    mb_method="FastIAMB",
    max_depth=3,
    alpha=0.01,
    max_K=5
)
discrete_pred_pag = discrete_result["PAG.DataFrame"]
discrete_eval = local_mark_evaluation(discrete_pred_pag, true_pag, target)

discrete_summary = pd.Series(
    {
        **discrete_eval,
        "CI_num": discrete_result["CI_num"],
        "runtime_sec": discrete_result["runtime_sec"],
    },
    name="discrete",
)
discrete_summary.to_frame()

,discrete
Local-SHD,0.000000
Mark-Precision,1.000000
Mark-Recall,1.000000
Mark-F1,1.000000
CI_num,196.000000
runtime_sec,0.020088


In [44]:
discrete_result_complete = data_MMB_by_MMB(
    discrete_observed_data,
    target=target,
    ci_method="chisq",
    mb_method="FastIAMB",
    max_depth=3,
    alpha=0.01,
    whether_fast=False,
    max_K=5
)
discrete_pred_pag_complete = discrete_result_complete["PAG.DataFrame"]
discrete_eval_complete = local_mark_evaluation(discrete_pred_pag_complete, true_pag, target)

discrete_summary = pd.Series(
    {
        **discrete_eval_complete,
        "CI_num": discrete_result_complete["CI_num"],
        "runtime_sec": discrete_result_complete["runtime_sec"],
    },
    name="discrete",
)
discrete_summary.to_frame()

,discrete
Local-SHD,0.000000
Mark-Precision,1.000000
Mark-Recall,1.000000
Mark-F1,1.000000
CI_num,274.000000
runtime_sec,0.021298


## 9. Oracle Learning

In [45]:
oracle_result = oracle_MMB_by_MMB(
    dag,
    target=target,
    latent_nodes=latent_nodes
)
oracle_pred_pag = oracle_result["PAG.DataFrame"]
oracle_eval = local_mark_evaluation(oracle_pred_pag, true_pag, target)
oracle_summary = pd.Series(
    {
        **oracle_eval,
        "CI_num": oracle_result["CI_num"],
        "runtime_sec": oracle_result["runtime_sec"],
    },
    name="oracle",
)
oracle_summary.to_frame()

,oracle
Local-SHD,0.00000
Mark-Precision,1.00000
Mark-Recall,1.00000
Mark-F1,1.00000
CI_num,150.00000
runtime_sec,0.01733
